In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import pybedtools

In [2]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
HIMORNA_INITIAL = DATA_INITIAL / "himorna"
HIMORNA = DATA / "himorna"
HM_INITIAL = HIMORNA_INITIAL / "histone_modifications"
HM = HIMORNA / "histone_modifications"
METHYL_INITIAL = HIMORNA_INITIAL / "methylation"
METHYL = HIMORNA / "methylation"
FANTOM = DATA / "fantom_cat"
PREPROCESSING = Path("../preprocessing")
HIMORNA_INDEX = PREPROCESSING / "himorna_index_new"
HIMORNA_GENES_SIGN =  PREPROCESSING / "himorna_genes_sign_new"
ID_HISTORY_CONVERTER_RES = DATA / "id_history_converter_res"

In [3]:
for directory in DATA, HIMORNA_INITIAL, HIMORNA:
    directory.mkdir(exist_ok=True, parents=True)

## Переаннотирование только уникальных пиков в pd.concat(hm_peaksm, methyl_peaks)

In [4]:
def get_peaks_coord(to_peaks):
    himorna_peaks = pd.read_csv(
            to_peaks,
            chunksize=100000000,
            sep="\t",
            usecols=["chrom", "start", "end"]
        )
    himorna_peaks = pd.concat(himorna_peaks)    
    himorna_peaks_coord = himorna_peaks[["chrom", "start", "end"]].drop_duplicates(ignore_index=True)
    return himorna_peaks_coord

In [5]:
def count_genome_coverage(anno):
    anno_bed_sort = pybedtools.BedTool.from_dataframe(anno).sort()
    genome_coverage = anno_bed_sort.genome_coverage(bga=True, genome='hg38')
    genome_coverage = genome_coverage.to_dataframe()
    genome_coverage["segment_lenght"] = genome_coverage["end"] - genome_coverage["start"]
    res = genome_coverage[genome_coverage["name"] != 0]["segment_lenght"].sum()
    return round(res / 3209286105 * 100, 2) # 3209286105 bp - размер генома человека

In [6]:
to_hm_peaks = HM_INITIAL / "Himorna_all.tsv.gz"
to_methyl_peaks = METHYL / "methylation_peaks.tsv.gz"

In [7]:
peaks_coord = {}
for table in [("hm_peaks",to_hm_peaks), ("methyl_peaks",to_methyl_peaks)]:
    print(table[0])
    peaks_coord[table[0]] = get_peaks_coord(to_peaks=table[1])

hm_peaks
methyl_peaks


In [8]:
[len(peaks_coord[i]) for i in peaks_coord]

[3968185, 27405966]

In [9]:
27405966 / 3968185 #Пиков метилирования больше в 7 раз, по сравнению с пиками гистоновых модификаций

6.906423465639833

In [10]:
himorna_peaks_coord = pd.concat([peaks_coord["hm_peaks"], peaks_coord["methyl_peaks"]], ignore_index=True)
himorna_peaks_coord.shape

(31374151, 3)

In [11]:
assert 3968185 + 27405966 == len(himorna_peaks_coord)

In [12]:
himorna_peaks_coord = himorna_peaks_coord.drop_duplicates(ignore_index=True)
himorna_peaks_coord.shape

(31374151, 3)

In [13]:
peaks_coord["hm_peaks"]["chrom"].nunique()

134

In [14]:
peaks_coord["methyl_peaks"]["chrom"].nunique()

25

In [15]:
himorna_peaks_coord["chrom"].nunique()

134

In [16]:
himorna_peaks_coord

,chrom,start,end
0,chr18,10441312,10441530
1,chr18,10441611,10441762
2,chr18,10442237,10443200
3,chr18,10443212,10443546
4,chr18,10443735,10444053
...,...,...,...
31374146,chrX,103029665,103029666
31374147,chrX,119678095,119678096
31374148,chrX,150307844,150307845
31374149,chr8,45570918,45570919


In [82]:
canonical_chrom = ["chr"+str(i) for i in range(1,23)]
canonical_chrom.append("chrX")
canonical_chrom.append("chrY")
canonical_chrom.append("chrM")
len(canonical_chrom)

25

In [26]:
mask = himorna_peaks_coord["chrom"].isin(canonical_chrom)
himorna_peaks_coord_canon_chrom = himorna_peaks_coord[mask]
himorna_peaks_coord_canon_chrom.shape

(31369304, 3)

31374151 - 31369304 #Потеряли не так много пиков, ничего страшного

In [25]:
count_genome_coverage(anno=himorna_peaks_coord_canon_chrom)

79.73

In [27]:
himorna_peaks_coord_canon_chrom_sortbed = pybedtools.BedTool.from_dataframe(himorna_peaks_coord_canon_chrom).sort()

In [28]:
to_file = FANTOM / 'fantom_l2_genes-hg38-sort-ensg_fixed.bed'
fantom_genes_hg38 = pd.read_csv(to_file, sep="\t", header=None, names=["chrom","start","end","name"], usecols=[0,1,2,3])
fantom_genes_hg38.head(3)

,chrom,start,end,name
0,chr1,159537,162485,CATG00000042730-gene
1,chr1,273882,274416,CATG00000042731-gene
2,chr1,588120,634563,CATG00000042732-gene


In [29]:
len(fantom_genes_hg38)

121781

In [104]:
fantom_genes_hg38[fantom_genes_hg38["name"] == "CATG00000006156-gene"]

,chrom,start,end,name
19476,chr11,74390265,74394560,CATG00000006156-gene


In [30]:
fantom_genes_hg38_sortbed = pybedtools.BedTool.from_dataframe(fantom_genes_hg38).sort()

In [32]:
himorna_peaks_coord_assoc_sortbed = fantom_genes_hg38_sortbed.window(himorna_peaks_coord_canon_chrom_sortbed, w=1000).sort()

In [39]:
himorna_peaks_coord_assoc_sortbed.head(5)

chr1	273882	274416	CATG00000042731-gene	chr1	273897	273898
 chr1	273882	274416	CATG00000042731-gene	chr1	273896	273897
 chr1	588120	634563	CATG00000042732-gene	chr1	612167	612168
 chr1	588120	634563	CATG00000042732-gene	chr1	610710	610711
 chr1	588120	634563	CATG00000042732-gene	chr1	612124	612125
 

In [34]:
%%time
len(himorna_peaks_coord_assoc_sortbed)

CPU times: user 5min 9s, sys: 8.06 s, total: 5min 17s
Wall time: 5min 6s


36907835

In [ ]:
37744393

In [35]:
%%time
himorna_peaks_coord_assoc_sortbed.count()

KeyboardInterrupt: 

In [41]:
himorna_peaks_coord_assoc_sortbed_v2 = himorna_peaks_coord_canon_chrom_sortbed.window(fantom_genes_hg38_sortbed, w=1000).sort()

In [42]:
himorna_peaks_coord_assoc_sortbed_v2.head(3)

chr1	273896	273897	chr1	273882	274416	CATG00000042731-gene
 chr1	273897	273898	chr1	273882	274416	CATG00000042731-gene
 chr1	587352	587353	chr1	588120	634563	CATG00000042732-gene
 

In [43]:
himorna_peaks_coord_assoc_sortbed_v2.count()

36907835

Так числа совпали - .window работает корректно. Результат идентичен что при расширении пиков и поиске пересечений с генами, что при расширении генов и поиске пересечений с пиками

In [44]:
himorna_peaks_coord_assoc = pybedtools.BedTool.to_dataframe(himorna_peaks_coord_assoc_sortbed, disable_auto_names=True, names=["gene_chrom", "gene_start", "gene_end", "gene_ens", "chrom", "start", "end"])
himorna_peaks_coord_assoc.head(3)

,gene_chrom,gene_start,gene_end,gene_ens,chrom,start,end
0,chr1,273882,274416,CATG00000042731-gene,chr1,273897,273898
1,chr1,273882,274416,CATG00000042731-gene,chr1,273896,273897
2,chr1,588120,634563,CATG00000042732-gene,chr1,612167,612168


In [45]:
len(himorna_peaks_coord_assoc)

36907835

In [118]:
himorna_peaks_coord_assoc[himorna_peaks_coord_assoc["start"] == 633422]

,gene_chrom,gene_start,gene_end,gene_ens,chrom,start,end
137,chr1,588120,634563,CATG00000042732-gene,chr1,633422,633423
368,chr1,629083,634376,ENSG00000248527-gene,chr1,633422,633423
521,chr1,631193,634926,ENSG00000198744-gene,chr1,633422,633423
589,chr1,631381,632616,ENSG00000237973-gene,chr1,633422,633423
643,chr1,632814,633438,ENSG00000229344-gene,chr1,633422,633423
712,chr1,633535,633741,ENSG00000240409-gene,chr1,633422,633423


In [101]:
himorna_peaks_coord_assoc[(himorna_peaks_coord_assoc["chrom"] == "chr11") & (himorna_peaks_coord_assoc["start"] == "74366424") \
                        & (himorna_peaks_coord_assoc["end"] == "74398020")]

,gene_chrom,gene_start,gene_end,gene_ens,chrom,start,end


In [102]:
himorna_peaks_coord_assoc[(himorna_peaks_coord_assoc["gene_ens"] == "CATG00000006156")]

,gene_chrom,gene_start,gene_end,gene_ens,chrom,start,end


## Загрузка, конкатенация и аннотация оригинальных таблиц

In [46]:
to_hm_peaks = HM_INITIAL / "Himorna_all.tsv.gz"
to_methyl_peaks = METHYL / "methylation_peaks.tsv.gz"

In [69]:
hm_peaks = pd.read_csv(
        to_hm_peaks,
        chunksize=100000000,
        sep="\t",
    )
hm_peaks = pd.concat(hm_peaks)
print("hm_peaks: done")

hm_peaks: done


In [126]:
hm_peaks.shape

(67288977, 8)

In [124]:
hm_peaks.groupby(by="hm").agg({"lnc_ens":pd.Series.nunique, "gene_ens":pd.Series.nunique})

KeyError: "Column(s) ['gene_ens'] do not exist"

In [71]:
hm_peaks[(hm_peaks["lnc"] == "MAPK6-DT") & (hm_peaks["hm"] == "H4K20me1") & (hm_peaks["peak"] == "peak_38549") & (hm_peaks["chrom"] == "chr11")]

,Unnamed: 0,hm,lnc,lnc_ens,peak,chrom,start,end,gene,gene_ens,corr
104022425,104022425,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,LOC100287896,ENSG00000254837,-0.891363
104022426,104022426,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,MIR548AL,ENSG00000264402,-0.891363
104022427,104022427,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,NaN,CATG00000006156,-0.891363
104022428,104022428,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,AP001372.1,ENSG00000254631,-0.891363
104022429,104022429,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,PGM2L1,ENSG00000165434,-0.891363
104022430,104022430,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,NaN,CATG00000002919,-0.891363


In [72]:
columns = ["hm","lnc","lnc_ens","peak","chrom","start","end","corr"]

In [73]:
hm_peaks = hm_peaks[columns]
hm_peaks

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511
3,H3K27ac,AGAP11,ENSG00000271880,peak_389211,chr18,10441312,10441530,0.491645
4,H3K27ac,AC015961.2,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911
...,...,...,...,...,...,...,...,...
104264739,H4K20me1,MAPK6-DT,ENSG00000259438,peak_18626,chr1,222815712,222856635,-0.730043
104264740,H4K20me1,AL158825.2,ENSG00000229587,peak_18626,chr1,222815712,222856635,-0.755635
104264741,H4K20me1,AL158825.2,ENSG00000229587,peak_18626,chr1,222815712,222856635,-0.755635
104264742,H4K20me1,AL158825.2,ENSG00000229587,peak_18626,chr1,222815712,222856635,-0.755635


In [74]:
hm_peaks = hm_peaks.drop_duplicates()
len(hm_peaks)

67288977

In [75]:
hm_peaks.head(3)

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511


In [76]:
hm_peaks[(hm_peaks["lnc"] == "MAPK6-DT") & (hm_peaks["hm"] == "H4K20me1") & (hm_peaks["peak"] == "peak_38549") & (hm_peaks["chrom"] == "chr11")]

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
104022425,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363


In [90]:
hm_peaks["lnc_ens"] = hm_peaks["lnc_ens"].astype('str')

/tmp/ipykernel_24898/2384402353.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hm_peaks["lnc_ens"] = hm_peaks["lnc_ens"].astype('str')


In [91]:
mask = hm_peaks["chrom"].isin(canonical_chrom)
hm_peaks_canon_chrom = hm_peaks[mask]
hm_peaks_canon_chrom.head(3)

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511


In [92]:
hm_peaks_canon_chrom.shape

(67014817, 8)

In [93]:
mask = hm_peaks["chrom"].isin(canonical_chrom)
hm_peaks_noncanon_chrom = hm_peaks[~mask]
hm_peaks_noncanon_chrom.head(3)

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
3891898,H3K27ac,AC053503.5,ENSG00000280323,peak_180459,chr11_KI270721v1_random,21849,22342,0.584858
3891899,H3K27ac,AC126696.3,ENSG00000260498,peak_180459,chr11_KI270721v1_random,21849,22342,-0.490055
3891900,H3K27ac,AC022336.2,ENSG00000260391,peak_180459,chr11_KI270721v1_random,21849,22342,0.477395


In [94]:
hm_peaks_noncanon_chrom.shape

(274160, 8)

In [77]:
methyl_peaks = pd.read_csv(
        to_methyl_peaks,
        chunksize=100000000,
        sep="\t",
    )
methyl_peaks = pd.concat(methyl_peaks)
print("methyl_peaks: done")

methyl_peaks: done


In [78]:
methyl_peaks.shape

(452430340, 7)

In [99]:
methyl_peaks

,hm,lnc_ens,peak,chrom,start,end,corr
0,methylation,ENSG00000099869,peak_10386212,chr12,49123587,49123588,-0.888
1,methylation,ENSG00000099869,peak_49210615,chr8,142304068,142304069,0.911
2,methylation,ENSG00000103472,peak_558140,chr1,15837335,15837336,-0.870
3,methylation,ENSG00000103472,peak_1734200,chr1,66118486,66118487,0.864
4,methylation,ENSG00000103472,peak_3431110,chr1,199066625,199066626,0.878
...,...,...,...,...,...,...,...
452430335,methylation,ENSG00000261222,peak_21156386,chr17,81669387,81669388,-0.897
452430336,methylation,ENSG00000261222,peak_28291839,chr2,222626043,222626044,-0.894
452430337,methylation,ENSG00000261222,peak_33936371,chr3,120774402,120774403,-0.938
452430338,methylation,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890


In [80]:
methyl_peaks["lnc_ens"].nunique()

1705

In [95]:
# Для метилирования не отбираем по хромосомам, поскольку там уже сожержатся только канонические

hm_peaks_canon_chrom["corr"] = np.round(hm_peaks_canon_chrom["corr"], 3)

In [87]:
methyl_peaks["lnc_ens"] = methyl_peaks["lnc_ens"].astype('str')

In [88]:
methyl_peaks["hm"] = "methylation"

In [96]:
columns = ["hm","lnc_ens","peak","chrom","start","end","corr"]

In [97]:
methyl_peaks = methyl_peaks[columns]

In [98]:
methyl_peaks = methyl_peaks.drop_duplicates()
len(methyl_peaks)

452430340

In [109]:
himorna_peaks = pd.concat([hm_peaks, methyl_peaks], ignore_index=True)

In [110]:
himorna_peaks.head(1)

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356


In [111]:
himorna_peaks.tail(1)

,hm,lnc,lnc_ens,peak,chrom,start,end,corr
519719316,methylation,NaN,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879


In [112]:
himorna_peaks.shape

(519719317, 8)

In [113]:
himorna_peaks_assoc = pd.merge(left=himorna_peaks, right=himorna_peaks_coord_assoc, left_on=["chrom", "start", "end"], right_on=["chrom", "start", "end"], how="left")

In [116]:
himorna_peaks_assoc

,hm,lnc,lnc_ens,peak,chrom,start,end,corr,gene_chrom,gene_start,gene_end,gene_ens
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356,NaN,NaN,NaN,NaN
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105,NaN,NaN,NaN,NaN
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511,NaN,NaN,NaN,NaN
3,H3K27ac,AGAP11,ENSG00000271880,peak_389211,chr18,10441312,10441530,0.491645,NaN,NaN,NaN,NaN
4,H3K27ac,AC015961.2,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
696878886,methylation,NaN,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,chr8,9551766.0,9788424.0,ENSG00000173273-gene
696878887,methylation,NaN,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,chr8,9740848.0,9741768.0,ENSG00000207701-gene
696878888,methylation,NaN,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,chr8,10712751.0,10839868.0,ENSG00000171056-gene
696878889,methylation,NaN,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,chr8,10725399.0,10839868.0,ENSG00000258724-gene


In [117]:
len(himorna_peaks), len(himorna_peaks_coord_assoc)

(519719317, 36907835)

In [119]:
to_save =  HIMORNA / "himorna_peaks-hm_meth-assoc.tsv.gz"
himorna_peaks_assoc.to_csv(to_save, sep="\t", compression="gzip", index=False)

In [28]:
to_file =  HIMORNA / "himorna_peaks-hm_meth-assoc.tsv.gz"
himorna_peaks_assoc = pd.read_csv(to_file, sep="\t")

In [122]:
himorna_peaks_assoc

,hm,lnc,lnc_ens,peak,chrom,start,end,corr,gene_chrom,gene_start,gene_end,gene_ens
0,H3K27ac,AP005060.1,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356,NaN,NaN,NaN,NaN
1,H3K27ac,AC012409.3,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105,NaN,NaN,NaN,NaN
2,H3K27ac,AC116651.1,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511,NaN,NaN,NaN,NaN
3,H3K27ac,AGAP11,ENSG00000271880,peak_389211,chr18,10441312,10441530,0.491645,NaN,NaN,NaN,NaN
4,H3K27ac,AC015961.2,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
696878886,methylation,NaN,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,chr8,9551766.0,9788424.0,ENSG00000173273-gene
696878887,methylation,NaN,ENSG00000261222,peak_47069123,chr8,9741473,9741474,-0.890000,chr8,9740848.0,9741768.0,ENSG00000207701-gene
696878888,methylation,NaN,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,chr8,10712751.0,10839868.0,ENSG00000171056-gene
696878889,methylation,NaN,ENSG00000261222,peak_47097647,chr8,10827627,10827628,-0.879000,chr8,10725399.0,10839868.0,ENSG00000258724-gene


In [123]:
himorna_peaks_assoc[(himorna_peaks_assoc["lnc"] == "MAPK6-DT") & (himorna_peaks_assoc["hm"] == "H4K20me1") & (himorna_peaks_assoc["peak"] == "peak_38549") & (himorna_peaks_assoc["chrom"] == "chr11")]

,hm,lnc,lnc_ens,peak,chrom,start,end,corr,gene_chrom,gene_start,gene_end,gene_ens
106296720,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74289292.0,74528886.0,CATG00000002919-gene
106296721,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74329976.0,74398392.0,ENSG00000165434-gene
106296722,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74351444.0,74485742.0,ENSG00000254631-gene
106296723,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74366871.0,74511511.0,ENSG00000254837-gene
106296724,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74390265.0,74394560.0,CATG00000006156-gene
106296725,H4K20me1,MAPK6-DT,ENSG00000259438,peak_38549,chr11,74366424,74398020,-0.891363,chr11,74397766.0,74401595.0,ENSG00000264402-gene


to_file = f"{ID_HISTORY_CONVERTER_RES}/bad_genes-resolutions.tsv"
df_bad_genes_resolutions_final = pd.read_csv(to_file, sep="\t")

dict_bad_genes_resolutions_final = {i:j for i,j in zip(df_bad_genes_resolutions_final["name"], df_bad_genes_resolutions_final["name_latest"])}
len(dict_bad_genes_resolutions_final)

dict_bad_genes_resolutions_final

lnc_ens_list = himorna_peaks_assoc["lnc_ens"].unique()
len(lnc_ens_list)

len((set(lnc_ens_list) & set(list(dict_bad_genes_resolutions_final.keys()))))# лучше просто перенести их на последнюю аннотацию генкода. Также как в статье Гришы Рябухина, черещ индекс Жаккара

himorna_peaks_assoc["gene_ens"] = himorna_peaks_assoc["gene_ens"].astype(str)

himorna_peaks_assoc["gene_ens-updated"] = himorna_peaks_assoc["gene_ens"].apply(lambda x: dict_bad_genes_resolutions_final[x.split("-")[0]]+"-gene" if x.split("-")[0] in dict_bad_genes_resolutions_final else x)

In [120]:
himorna_peaks_assoc.groupby(by="hm").agg({"lnc_ens":pd.Series.nunique, "gene_ens":pd.Series.nunique})

,lnc_ens,gene_ens
hm,,
H3K27ac,2031,107977
H3K27me3,1401,90337
H3K36me3,1321,100325
H3K4me1,1954,113410
H3K4me2,1031,87055
H3K4me3,1752,96787
H3K79me2,802,63039
H3K9ac,1104,44435
H3K9me3,1079,79152
